# 🧠 EX56: Prediction Sources

`model.predict(source=...)` auto-selects a data loader:

| Source | Example | Loader |
|--------|---------|--------|
| File path | `"img.jpg"` | PIL/OpenCV |
| Directory | `"imgs/"` | glob+sort |
| URL | `"https://..."` | urllib |
| NumPy (HWC, BGR) | `np.ndarray` | direct buffer |
| PIL Image | `Image.open(...)` | numpy convert |
| Webcam | `0` | cv2.VideoCapture |
| RTSP | `"rtsp://..."` | cv2.VideoCapture |
| Video | `"video.mp4"` | cv2.VideoCapture |

**Pre-processing:** decode → letterbox → normalize [0→1] → CHW tensor

## 🔗 Links
- [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os
import cv2, torch, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
from solution import run_inference
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

# Synthetic test image
img_np = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.putText(img_np, "YOLO", (80, 240), cv2.FONT_HERSHEY_SIMPLEX, 3, (255,255,255), 4)
cv2.imwrite("src_test.jpg", img_np)

print("\n--- AUDIT & INSPECTION START ---")
model = YOLO("yolo11n.pt")

print("[Source 1] File path...")
r1 = run_inference("yolo11n.pt", "src_test.jpg", save_results=True)
print(f"  Detections: {len(r1[0].boxes)}")

print("[Source 2] NumPy array (BGR HWC)...")
r2 = model.predict(source=img_np, verbose=False)
print(f"  Detections: {len(r2[0].boxes)}  | input shape={img_np.shape} dtype={img_np.dtype}")

print("[Source 3] PIL Image (RGB)...")
pil_img = Image.open("src_test.jpg")
r3 = model.predict(source=pil_img, verbose=False)
print(f"  Detections: {len(r3[0].boxes)}")
print("--- AUDIT & INSPECTION END ---")

fig, axes = plt.subplots(1, 3, figsize=(14,4))
for ax, (title, res) in zip(axes, [("File path",r1[0]),("NumPy",r2[0]),("PIL",r3[0])]):
    ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(title); ax.axis("off")
plt.suptitle("Same model — different source types"); plt.tight_layout(); plt.show()

del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
if os.path.exists("src_test.jpg"): os.remove("src_test.jpg")
